# 03 — Pilot Run (24 samples × 6 stages, Qwen2.5-Omni-7B)

Sanity-check the full pipeline before committing to the main run. Goals:
- Verify Qwen2.5-Omni-7B loads, processes audio + video, and returns parseable output.
- Verify the inline attribution probe in S3 produces parseable rankings >= 80% of the time. If parse rate is bad, iterate on prompt wording HERE before the main run.
- Spot-check that S0 ≈ random (~25%), S2 << S3 on audio-essential tasks, etc.
- Measure per-stage seconds/sample so we can size the main run's compute budget.

**Runtime:** ~20 minutes on H100 (4 samples/task × 6 tasks × 6 stages × ~10s ≈ 24 minutes).


In [ ]:
import os, sys, json, time, random
REPO = '/content/omnimodel-research'
if os.path.exists(REPO):
    %cd $REPO
    sys.path.insert(0, REPO)

with open('data/eval_samples_clean.json') as f:
    all_samples = json.load(f)
print(f'Available eval samples: {len(all_samples)}')

# Pilot: 4 per task type from the clean set
from src.data_utils import balanced_subsample
pilot_samples = balanced_subsample(all_samples, n_per_task=4, seed=7)
print(f'Pilot set: {len(pilot_samples)} samples')

with open('data/pilot_samples.json', 'w') as f:
    json.dump(pilot_samples, f, indent=2)


In [ ]:
from src.model_utils import OmniModelWrapper, TextOnlyModelWrapper
from src.prompts import STAGE_BUILDERS
from src.parse_utils import parse_answer_confidence, parse_attribution, parse_reason

print('Loading Qwen2.5-Omni-7B...')
omni = OmniModelWrapper('Qwen/Qwen2.5-Omni-7B')
print('Loading Qwen2.5-1.5B-Instruct (text baseline)...')
text_only = TextOnlyModelWrapper('Qwen/Qwen2.5-1.5B-Instruct')
print('Models loaded.')


In [ ]:
# Helpers to construct stage-specific inputs
from src.transcript_utils import load_transcript

def stage_inputs(stage, sample):
    vid = sample['video_id']
    paths = {
        'video': f'data/videos/{vid}.mp4',
        'audio': f'data/audio/{vid}.wav',
        'silent': f'data/silent_video/{vid}_silent.mp4',
        'transcript': f'data/transcripts/{vid}.txt',
        # Mismatched transcript is keyed by qa_id (a single video may have
        # multiple QAs that need DIFFERENT mismatched donors)
        'mismatched': f'data/transcripts_mismatched/{sample["qa_id"]}.txt',
    }
    if stage == 'S0':
        return {'video_path': None, 'audio_path': None, 'extra': {}}
    if stage == 'S1':
        return {'video_path': None, 'audio_path': paths['audio'], 'extra': {}}
    if stage == 'S2':
        return {'video_path': paths['silent'], 'audio_path': None, 'extra': {}}
    if stage == 'S3':
        return {'video_path': paths['video'], 'audio_path': paths['audio'], 'extra': {}}
    if stage == 'S4':
        return {'video_path': paths['video'], 'audio_path': paths['audio'],
                'extra': {'transcript': load_transcript(paths['transcript'])}}
    if stage == 'S5':
        return {'video_path': paths['video'], 'audio_path': paths['audio'],
                'extra': {'mismatched_transcript': load_transcript(paths['mismatched'])}}
    raise ValueError(stage)

def run_one(stage, sample, model):
    inp = stage_inputs(stage, sample)
    prompt = STAGE_BUILDERS[stage](sample['question'], sample['options'], **inp['extra'])
    resp = model.generate(prompt, video_path=inp['video_path'], audio_path=inp['audio_path'])
    answer, conf = parse_answer_confidence(resp.text)
    attr = parse_attribution(resp.text) if stage == 'S3' else None
    reason = parse_reason(resp.text) if stage == 'S3' else None
    return {
        'qa_id': sample['qa_id'],
        'video_id': sample['video_id'],
        'task_type': sample['task_type'],
        'task_code': sample.get('task_code'),
        'stage': stage,
        'ground_truth': sample['answer'],
        'predicted_answer': answer,
        'verbalized_confidence': conf,
        'answer_logprobs': resp.answer_logprobs,
        'attribution': attr,
        'attribution_reason': reason,
        'raw_response': resp.text,
    }


In [ ]:
import os
os.makedirs('results/raw_predictions/pilot', exist_ok=True)

STAGES = ['S0', 'S1', 'S2', 'S3', 'S4', 'S5']
timings = {}

for stage in STAGES:
    print(f'\n=== {stage} ===')
    t0 = time.time()
    records = []
    model = text_only if stage == 'S0' else omni
    for i, s in enumerate(pilot_samples):
        try:
            rec = run_one(stage, s, model)
        except Exception as e:
            rec = {'qa_id': s['qa_id'], 'video_id': s['video_id'],
                   'task_type': s['task_type'], 'task_code': s.get('task_code'),
                   'stage': stage, 'ground_truth': s['answer'],
                   'predicted_answer': None, 'verbalized_confidence': None,
                   'answer_logprobs': None, 'attribution': None,
                   'attribution_reason': None, 'raw_response': f'<ERROR: {e}>'}
        records.append(rec)
        if (i + 1) % 5 == 0:
            print(f'  [{i+1}/{len(pilot_samples)}]')

    with open(f'results/raw_predictions/pilot/{stage}.json', 'w') as f:
        json.dump(records, f, indent=2)
    elapsed = time.time() - t0
    timings[stage] = {'total_s': elapsed, 'per_sample_s': elapsed / len(pilot_samples)}
    print(f'  {stage} done in {elapsed:.0f}s ({elapsed/len(pilot_samples):.1f}s/sample)')

with open('results/raw_predictions/pilot/_timings.json', 'w') as f:
    json.dump(timings, f, indent=2)


## Sanity-check the pilot results

In [ ]:
from src.metrics import accuracy_per_task

for stage in STAGES:
    with open(f'results/raw_predictions/pilot/{stage}.json') as f:
        recs = json.load(f)
    parsed = sum(1 for r in recs if r['predicted_answer'] is not None)
    acc = accuracy_per_task(recs)
    print(f'{stage}: parse_rate={parsed}/{len(recs)}  overall_acc={acc.get("OVERALL", 0):.2f}')
    if stage == 'S3':
        attr_parsed = sum(1 for r in recs if r['attribution'] is not None)
        print(f'   S3 attribution parse rate: {attr_parsed}/{len(recs)}')


## Decisions before main run

After running the cells above, check:

1. **Parse rates** — if any stage has < 80% answer parse rate, fix the prompt in `src/prompts.py` before kicking off the main run.
2. **S3 attribution parse rate** — < 80% means the attribution probe needs prompt iteration. The format `[ANSWER] X [CONFIDENCE] Y / [RANK] Audio=A, Visual=V, Text=T` is brittle.
3. **Per-sample timings** — if S3/S4/S5 are >> 15s/sample, the 120-sample main run will exceed the 2hr budget. Consider reducing N_PER_TASK to 15 in 04.
4. **S0 accuracy ≈ 0.25** — if S0 is much higher, the AVUT text-shortcut filtering is being violated by the small text model. Worth flagging in the paper either way.

When all checks pass, proceed to `04_main_eval_qwen.ipynb`.
